In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
import warnings

# Отключаем предупреждения
warnings.filterwarnings('ignore')

In [10]:
# Загружаем данные
df = pd.read_csv('cleaned_data.csv')

In [11]:
# Удаляем выбросы для целевой переменной CC50
Q1 = df['CC50, mM'].quantile(0.25)
Q3 = df['CC50, mM'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

# Оставляем только те строки, где CC50 не превышает верхнюю границу выбросов
df = df[df['CC50, mM'] <= upper_bound]
print(f"Размер данных после удаления выбросов CC50: {df.shape}")

Размер данных после удаления выбросов CC50: (902, 89)


In [12]:
# Убираем все целевые переменные и классы, чтобы модель не могла подсмотреть ответ
targets_to_drop = ['IC50, mM', 'CC50, mM', 'SI', 'IC50_class', 'CC50_class', 'SI_median_class', 'SI_8_class']
X = df.drop(columns=targets_to_drop, errors='ignore')
y = df['CC50, mM']

# Разбиваем данные на выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
# Масштабируем данные
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
# Задаем словарь с моделями
models = {
    "Ridge (Линейная регрессия)": (Ridge(), {'alpha': [100.0, 200.0]}),
    "Random Forest (Случайный лес)": (RandomForestRegressor(random_state=42), {'n_estimators': [50, 100], 'max_depth': [None, 10]}),
    "Gradient Boosting (Градиентный бустинг)": (GradientBoostingRegressor(random_state=42), {'n_estimators': [100, 300, 500], 'learning_rate': [0.01, 0.05, 0.1]}),
    "SVR (Опорные векторы)": (SVR(), {'C': [50.0, 100.0]}),
    "KNN (Ближайшие соседи)": (KNeighborsRegressor(), {'n_neighbors': [10, 20]})
}

In [15]:
# Обучение и оценка моделей
results = {}

for name, (model, params) in models.items():
    grid = GridSearchCV(model, params, cv=3, scoring='r2', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)

    # Предсказание на лучшей модели
    best_model = grid.best_estimator_
    predictions = best_model.predict(X_test_scaled)

    # Расчет метрик
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)

    results[name] = {'R2': r2, 'MAE': mae, 'Best Params': grid.best_params_}

In [16]:
# Итоговые результаты
print("Итоговые результаты сравнения моделей (CC50):\n")
for name, metrics in results.items():
    print(f"Модель: {name}")
    print(f"Лучшие параметры: {metrics['Best Params']}")
    print(f"Метрика R2: {metrics['R2']:.3f}")
    print(f"Ошибка MAE: {metrics['MAE']:.3f}\n")

Итоговые результаты сравнения моделей (CC50):

Модель: Ridge (Линейная регрессия)
Лучшие параметры: {'alpha': 100.0}
Метрика R2: 0.302
Ошибка MAE: 335.738

Модель: Random Forest (Случайный лес)
Лучшие параметры: {'max_depth': 10, 'n_estimators': 100}
Метрика R2: 0.473
Ошибка MAE: 267.085

Модель: Gradient Boosting (Градиентный бустинг)
Лучшие параметры: {'learning_rate': 0.05, 'n_estimators': 100}
Метрика R2: 0.467
Ошибка MAE: 279.738

Модель: SVR (Опорные векторы)
Лучшие параметры: {'C': 100.0}
Метрика R2: 0.339
Ошибка MAE: 297.362

Модель: KNN (Ближайшие соседи)
Лучшие параметры: {'n_neighbors': 10}
Метрика R2: 0.381
Ошибка MAE: 297.049



**Вывод для CC50.**

В ходе прогнозирования CC50 лучший результат показал Случайный лес (R2 = 0.473, MAE = 267.1). Градиентный бустинг отработал почти на том же уровне (R2 = 0.467), модели KNN и SVR показали средние результаты с R2 около 0.34-0.38, а линейная регрессия (Ridge) снова справилась хуже всего.

**Применимость.** Ансамблевые модели на основе деревьев смогли выявить нелинейные зависимости в данных. Значение R2 почти достигло 0.5, что говорит об ограниченной применимости моделей: их можно использовать для грубой, предварительной оценки токсичности новых соединений (скрининга), но для точных медицинских выводов точности пока не хватает.

**Рекомендации по улучшению.** Помимо добавления новых данных и перехода к задаче классификации, здесь имеет смысл попробовать методы отбора признаков (Feature Selection), чтобы удалить зашумляющие дескрипторы и оставить только те, которые реально влияют на токсичность.